In [1]:
# Force install required libraries directly in the notebook environment
import sys
!{sys.executable} -m pip install torch torchvision opencv-python segment-anything pandas numpy requests tqdm mercantile pillow matplotlib seaborn scikit-learn xgboost

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.


In [2]:
import torch
import cv2
import numpy as np
import pandas as pd
import os
import gc
import sys
from tqdm import tqdm
from segment_anything import sam_model_registry, SamPredictor

# --- 1. HARDWARE SETUP (MAC OPTIMIZATION) ---
if torch.backends.mps.is_available():
    DEVICE = "mps"
    print("✅ Success: Using Apple MPS (Neural Engine)")
elif torch.cuda.is_available():
    DEVICE = "cuda"
    print("✅ Using NVIDIA GPU")
else:
    DEVICE = "cpu"
    print("⚠️ Warning: Running on CPU (Slower)")

# --- 2. MODEL PATH ---
# Auto-download weights if missing locally
CHECKPOINT_PATH = "../models/sam_vit_b_01ec64.pth"
MODEL_URL = "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth"

if not os.path.exists(CHECKPOINT_PATH):
    print("Downloading SAM weights...")
    os.makedirs("../models", exist_ok=True)
    import requests
    response = requests.get(MODEL_URL)
    with open(CHECKPOINT_PATH, "wb") as f:
        f.write(response.content)
    print("Download complete.")

# --- 3. LOAD MODEL ---
# Load to MPS device immediately
print("Loading SAM model to GPU...")
sam = sam_model_registry["vit_b"](checkpoint=CHECKPOINT_PATH)
sam.to(device=DEVICE)
predictor = SamPredictor(sam)
print("Model loaded and ready.")

✅ Success: Using Apple MPS (Neural Engine)
Loading SAM model to GPU...
Model loaded and ready.


In [3]:
# --- CONFIGURATION ---
# Paths relative to the 'notebooks' folder
IMAGE_DIR = "../data/images"
INPUT_CSV = "../data/processed/clean_homes.csv"
OUTPUT_CSV = "../data/processed/final_dataset.csv"
TENSOR_DIR = "../data/mask_tensors"

# Ensure output directories exist
os.makedirs(os.path.dirname(OUTPUT_CSV), exist_ok=True)
os.makedirs(TENSOR_DIR, exist_ok=True)

# Constants
PIXELS_TO_METERS = 0.07
CENTER_POINT = np.array([[320, 320]])
POINT_LABEL = np.array([1])

# --- HELPER FUNCTIONS ---

def get_house_mask(image_rgb):
    """Uses SAM to find the central structure."""
    # Predictor image is set in the main loop
    masks, scores, _ = predictor.predict(
        point_coords=CENTER_POINT,
        point_labels=POINT_LABEL,
        multimask_output=True
    )
    best_idx = np.argmax(scores)
    house_mask = masks[best_idx].astype(np.uint8)

    # Sanity Check: Reject green houses (Tree canopy errors)
    center_px = image_rgb[320, 320]
    # If Green is dominant over Red and Blue
    if center_px[1] > center_px[0] + 10 and center_px[1] > center_px[2] + 10:
        return np.zeros_like(house_mask)
    
    return house_mask

def get_vegetation_layers(image_rgb, house_mask):
    """Vectorized Color/Texture analysis (CPU/Numpy is fast enough here)."""
    # 1. ExG (Excess Green) Index
    r, g, b = image_rgb[:,:,0].astype(float), image_rgb[:,:,1].astype(float), image_rgb[:,:,2].astype(float)
    exg = 2*g - r - b
    
    green_mask = (exg > 10).astype(np.uint8)
    green_mask[house_mask == 1] = 0 
    
    if np.sum(green_mask) == 0:
        return np.zeros_like(green_mask), np.zeros_like(green_mask)

    # 2. Texture Analysis (Variance) to separate Trees vs Grass
    g_channel = image_rgb[:,:,1]
    mean = cv2.boxFilter(g_channel, cv2.CV_32F, (3, 3))
    sq_mean = cv2.boxFilter(g_channel**2, cv2.CV_32F, (3, 3))
    variance = sq_mean - (mean**2)
    
    hsv_v = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2HSV)[:,:,2]
    
    # Tree = Green AND (Rough Texture OR Dark Shadows)
    is_rough = variance > 200
    is_dark = hsv_v < 100
    
    tree_mask = np.zeros_like(green_mask)
    tree_mask[(green_mask == 1) & (is_rough | is_dark)] = 1
    
    # Grass = Green AND NOT Tree
    grass_mask = np.zeros_like(green_mask)
    grass_mask[(green_mask == 1) & (tree_mask == 0)] = 1
    
    # 3. Cleanup
    kernel = np.ones((3,3), np.uint8)
    tree_mask = cv2.morphologyEx(tree_mask, cv2.MORPH_OPEN, kernel, iterations=1)
    tree_mask = cv2.morphologyEx(tree_mask, cv2.MORPH_CLOSE, kernel, iterations=2)
    grass_mask = cv2.morphologyEx(grass_mask, cv2.MORPH_OPEN, kernel, iterations=1)
    
    return tree_mask, grass_mask

def compute_metrics(house_mask, tree_mask, grass_mask):
    """Calculates tabular features."""
    feats = {}
    
    # Areas
    feats['structure_area_m2'] = np.sum(house_mask) * (PIXELS_TO_METERS**2)
    feats['tree_area_m2'] = np.sum(tree_mask) * (PIXELS_TO_METERS**2)
    feats['grass_area_m2'] = np.sum(grass_mask) * (PIXELS_TO_METERS**2)
    
    # Counts
    nb_trees, _, _, _ = cv2.connectedComponentsWithStats(tree_mask, connectivity=8)
    feats['tree_count'] = max(0, nb_trees - 1)
    
    # Defensible Space
    if np.sum(house_mask) > 0 and np.sum(tree_mask) > 0:
        dist_map = cv2.distanceTransform(1-house_mask, cv2.DIST_L2, 3)
        min_dist_px = np.min(dist_map[tree_mask == 1])
        feats['defensible_space_m'] = min_dist_px * PIXELS_TO_METERS
    else:
        feats['defensible_space_m'] = 100.0
        
    # Compactness
    if np.sum(house_mask) > 0:
        contours, _ = cv2.findContours(house_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if contours:
            perim = cv2.arcLength(contours[0], True)
            area = cv2.contourArea(contours[0])
            feats['compactness'] = (4 * 3.14159 * area) / (perim**2 + 1e-6)
        else: feats['compactness'] = 0
    else: feats['compactness'] = 0
    
    return feats

def create_stack(house_mask, tree_mask):
    """Creates 3-Channel Tensor for CNN."""
    if np.sum(house_mask) > 0:
        dist_map = cv2.distanceTransform(1-house_mask, cv2.DIST_L2, 3)
        risk_channel = np.clip(255 - (dist_map * 0.6), 0, 255).astype(np.uint8)
        risk_channel = cv2.bitwise_and(risk_channel, risk_channel, mask=tree_mask)
    else:
        risk_channel = np.zeros_like(house_mask)
        
    stack = np.dstack([house_mask * 255, tree_mask * 255, risk_channel])
    stack_resized = cv2.resize(stack, (224, 224), interpolation=cv2.INTER_NEAREST)
    return stack_resized

In [4]:
# --- EXECUTION SETTINGS ---
BATCH_SIZE = 500

# 1. Map Existing Progress
# A. Check CSV
csv_ids = set()
if os.path.exists(OUTPUT_CSV):
    try:
        existing_df = pd.read_csv(OUTPUT_CSV)
        csv_ids = set(existing_df['id'].astype(str))
        print(f"📋 Found {len(csv_ids)} entries in CSV.")
    except:
        print("⚠️ CSV corrupt. Starting fresh.")
else:
    # Init Headers
    cols = ['id', 'target', 'lat', 'lon', 
            'structure_area_m2', 'tree_area_m2', 'grass_area_m2', 
            'tree_count', 'defensible_space_m', 'compactness']
    pd.DataFrame(columns=cols).to_csv(OUTPUT_CSV, index=False)
    print("🚀 Created new CSV.")

# B. Check Tensors
existing_tensors = set()
if os.path.exists(TENSOR_DIR):
    files = os.listdir(TENSOR_DIR)
    for f in files:
        if f.endswith('.npy'):
            existing_tensors.add(f.split('_')[0])
    print(f"📦 Found {len(existing_tensors)} tensors.")

# 2. Main Loop
if not os.path.exists(INPUT_CSV):
    print(f"⚠️ Input CSV not found at {INPUT_CSV}. Please run data_cleaning.ipynb first.")
else:
    df_meta = pd.read_csv(INPUT_CSV)
    df_meta['id_str'] = df_meta['id'].astype(str)
    results_buffer = []

    print(f"🔄 Scanning {len(df_meta)} homes...")

    for i, row in tqdm(df_meta.iterrows(), total=len(df_meta), unit="img"):
        uid = row['id_str']
        
        # Smart Checks
        needs_csv = uid not in csv_ids
        needs_tensor = uid not in existing_tensors
        
        # Optimization: Skip if both exist
        if not needs_csv and not needs_tensor:
            continue

        # Path check
        fname = f"{row['filename']}.jpg" if 'filename' in row and pd.notna(row['filename']) else f"{row['id']}.jpg"
        img_path = os.path.join(IMAGE_DIR, fname)
        
        if not os.path.exists(img_path):
            continue

        try:
            # A. Run Pipeline
            image = cv2.imread(img_path)
            if image is None: continue
            image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

            # Run Encoder (GPU Heavy Lift)
            predictor.set_image(image_rgb)
            
            # Get Masks
            house_mask = get_house_mask(image_rgb)
            tree_mask, grass_mask = get_vegetation_layers(image_rgb, house_mask)

            # B. Task 1: Generate CSV Data
            if needs_csv:
                feats = compute_metrics(house_mask, tree_mask, grass_mask)
                if feats:
                    record = {
                        'id': row['id'], 'target': row['target'],
                        'lat': row['lat'], 'lon': row['lon'],
                        **feats
                    }
                    results_buffer.append(record)
                    csv_ids.add(uid)

            # C. Task 2: Generate Tensor
            if needs_tensor:
                stack = create_stack(house_mask, tree_mask)
                save_name = f"{row['id']}_{int(row['target'])}.npy"
                np.save(os.path.join(TENSOR_DIR, save_name), stack)
                # existing_tensors.add(uid) # Optional

        except Exception as e:
            print(f"⚠️ Error on {fname}: {e}")
            continue

        # D. Batch Save (CSV)
        if len(results_buffer) >= BATCH_SIZE:
            pd.DataFrame(results_buffer).to_csv(OUTPUT_CSV, mode='a', header=False, index=False)
            results_buffer = []
            gc.collect()

    # Final Flush
    if results_buffer:
        pd.DataFrame(results_buffer).to_csv(OUTPUT_CSV, mode='a', header=False, index=False)

    print("\n✅ SYNC COMPLETE.")

📋 Found 21361 entries in CSV.
📦 Found 20940 tensors.
🔄 Scanning 22214 homes...


100%|██████████| 22214/22214 [00:00<00:00, 58224.70img/s]


✅ SYNC COMPLETE.
